
#Solving the Install Problem using SMT solvers

Every time a piece of software is installed, something has to work out which other
packages need to come along with it and which ones cannot sit on the same system.
In this notebook, we will look at SMT solvers and see how they can be used to answer
that question. Particularly, we will encode dependencies and conflicts between
packages as constraints on a set of boolean variables, and let the solver find an
installation profile that satisfies all of them at once.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, display_pkg_struct, display_pkg_solution, pkg_output_string
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Boolean

Suppose you have the following three boolean constraints and you want to check if there's a solution (an assignment of the variables) that satisfies all of them:

$$ x_1 \lor x_2 \lor x_3 $$

$$ \neg x_1 \implies \neg x_2$$

$$  x_1 \land x_3  $$

Let's see how we can do this using Z3.

In [ ]:
s = Solver() # initialize Z3 solver

# initialize variables

x1 = Bool('x_1') # declaring that x_1 is a boolean variable in Z3 which will be referred to as x1 in Python
x2 = Bool('x_2')
x3 = Bool('x_3')

# Note: we can also initialize multiple variables like so: x1, x2, x3 = Bools('x_1 x_2 x_3')

# we use s.add(.) to add a constraint to our solver s
# constraints can be made using many different operations such as Or, And, Not,
# equality, etc.

# here's how we would add the constraints above to our solver:

s.add( Or( x1, x2, x3 ) ) # add the first constraint
s.add( Implies( Not(x1), Not(x2) ) ) # add the second constraint
s.add( And( x1, x3 ) ) # add the third constraint

In [ ]:
# to view the constraints in our solver, we can use the following:
print( s )
# this prints the constraints as they appear in Z3 using Z3's notation

For better readability, this notebook also has a custom print function to view our constraints in LaTeX format, like so:

In [ ]:
showSolver( s )

In [ ]:
# we can use s.check() to run the solver and check whether its satisfiable:
print ( s.check() )

 "sat" means our system of constraints is satisfiable

In [ ]:
# after using s.check(),  we can use s.model() to output a solution if one exsits
solution = s.model()
print( solution )

Let's modify our system of constraint a bit and see if it's still satisfiable. Suppose we want to check if there's a solution where $x_1 = \neg x_3$. Let's see how we would do this with Z3.

In [ ]:
s.add( x1 == Not(x3) )
showSolver( s )

In [ ]:
print( s.check() ) # check if solution exists with new constraint

"unsat" means the system is not satisfiable, i.e., there is no assignment on the variables $x_1$, $x_2$, and $x_3$ that satisfies all the constraints we gave to the solver. **Note that if we were to run s.model() now we would get an error.**

# Solving the Install Problem

The **install problem** consists of determining whether a new set of packages can be installed in a system. This application is based on the article [OPIUM: Optimal Package Install/Uninstall Manager](http://cseweb.ucsd.edu/~rjhala/papers/opium.pdf). Many packages depend on other packages to provide some functionality. Each distribution contains a meta-data file that explicates the requirements of each package of the distribution. The meta-data contains details like the name, version, etc. More importantly, it contains **depends** and **conflicts** clauses that stipulate which other packages should be on the system. The depends clauses stipulate which other packages must be present. The conflicts clauses stipulate which other packages must not be present.

Let's see how we can solve this using Z3. The idea is to define a Boolean variable for each package. This variable is true if the package must be in the system. First, let's see a simple package structure example:

In [ ]:
a, b = Bools('a b')
s = Solver()
s.add(Implies(a, b), a)

display_pkg_struct(s)

Here, we have two packages `a` and `b`. The arrow pointing from `a` to `b` represents the dependency from `a` to `b`; if package `a` is to be installed, then package `b` must also be installed. The fact that `a` is shaded blue means that `a` is a requested package and must be installed to satisfy the problem. Let's look at another dependency example:

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')
s = Solver()
s.add(
    And(
      Implies(a, b),
      Implies(a, c),
      Implies(a, d),
      Implies(d, Or(e, f))),
    a
)

display_pkg_struct(s)

Here, `a` depends on `b`, `c`, and `d`. The joint dependency connnection below `d` means that `d` requires either `e` or `f` to be installed for it to be installed. To make our lives a bit easier, we'll define a helper function that easily lets us define these types of relations:

In [ ]:
def DependsOn(pack, deps):
    if is_expr(deps): # Allows us to do something like DependsOn(a, b) instead of DependsOn(a, [b])
        return Implies(pack, deps)
    else:
        return And([ Implies(pack, dep) for dep in deps ])

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')

s = Solver()
s.add(
    DependsOn(a, [b, c, d]),
    DependsOn(d, [Or(e, f)]),
    a
)

display_pkg_struct(s)

Up until now, we've just been looking at how to visualize these dependency structures, so now let's look at actually solving them:

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')

s = Solver()
s.add(
    DependsOn(a, [b, c, d]),
    DependsOn(d, [Or(e, f)]),
    a
)

display_pkg_solution(s)

Through the graph we can easily see a package installation profile that will satisfy our constaints; the packages shaded blue are specifically requested and so must be installed, and the packages shaded green have been chosen by the SAT solver to satisfy the given install problem. In the above example, installing packages `a`, `b`, `c`, `d`, and `f` is enough to satify the constraints. Package `e` is not needed.

Let's look at a slightly more complex example:

In [ ]:
a, b, c, d = Bools('a b c d')

s = Solver()
s.add(
    DependsOn(a, [b, c]),
    DependsOn(c, d),
    Or(Not(b), Not(d)),
    a
)

display_pkg_struct(s)

Here, the red line between `b` and `d` represents a conflict; if package `b` is to be installed, the package `d` must _not_ be installed. Let's try and define a helper function for this. Change the return line in the following function:

In [ ]:
def Conflict(p1, p2):
  return Or( p1, p2 ) # CHANGE THIS LINE

In [ ]:
a, b, c, d = Bools('a b c d')

s = Solver()
s.add(
    DependsOn(a, [b, c]),
    DependsOn(c, d),
    Conflict(b, d),
    a
)

display_pkg_struct(s)

Do you think the above package configuration is satisfiable? Let's see:

In [ ]:
display_pkg_solution(s)

This is because packages `b` and `d` are both necessary _and_ conflicting.

Run the cell below to produce a short string summarizing your solution. If your instructor has asked for it, copy and paste that string into your submission.

In [ ]:
pkg_output_string(s)


###Congratulations! You just used an SMT solver to decide which packages a system can install!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm